In [22]:
# USE %pip TO INSTALL DIRECTLY INTO THE ACTIVE KERNEL
%pip install lightgbm xgboost imbalanced-learn scikit-learn joblib

  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl.metadata (17 kB)
Using cached lightgbm-4.6.0-py3-none-win_amd64.whl (1.5 MB)
Note: you may need to restart the kernel to use updated packages.


# ADVANCED Multi-Model Training Pipeline: GLP-1 Adverse Event Seriousness
This notebook covers the end-to-end ML pipeline: Data Cleaning, Feature Engineering, Multi-Model Comparison, **Advanced Hyperparameter Tuning**, and Model Serialization.

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## Phase 1: Data Preprocessing

In [24]:
df = pd.read_csv('Dataset/adverse_events.csv')

# 1. Standardize Age to Years
def convert_age_to_years(row):
    age, unit = row['patient_age'], row['patient_age_unit']
    if pd.isna(age) or pd.isna(unit): return age
    if unit == 'Months': return age / 12
    elif unit == 'Days': return age / 365
    elif unit == 'Weeks': return age / 52
    elif unit == 'Decades': return age * 10
    return age

df['patient_age_years'] = df.apply(convert_age_to_years, axis=1)
df['patient_age_years'] = df.groupby('generic_name')['patient_age_years'].transform(lambda x: x.fillna(x.median()))
df['patient_age_years'].fillna(df['patient_age_years'].median(), inplace=True)

# 2. Clean Sex & Weight
df['patient_sex'].fillna('Unknown', inplace=True)
df['patient_weight_kg'] = df.groupby('generic_name')['patient_weight_kg'].transform(lambda x: x.fillna(x.median()))
df['patient_weight_kg'].fillna(df['patient_weight_kg'].median(), inplace=True)

# 3. Target Encoding
df['serious_encoded'] = df['serious'].apply(lambda x: 1 if x == 1 or str(x).lower() in ['yes', 'true', 'serious'] else 0)

# 4. Feature Engineering: Top 20 Reactions
top_reactions = df['reaction'].value_counts().nlargest(20).index.tolist()
for react in top_reactions:
    col_name = f"react_{react.lower().replace(' ', '_')}"
    df[col_name] = (df['reaction'] == react).astype(int)

# 5. Categorical Encoding
le_sex = LabelEncoder()
df['patient_sex_encoded'] = le_sex.fit_transform(df['patient_sex'])
drug_dummies = pd.get_dummies(df['generic_name'], prefix='drug')
df = pd.concat([df, drug_dummies], axis=1)

## Phase 2: Multi-Model Benchmark

In [25]:
feature_cols = ['patient_age_years', 'patient_weight_kg', 'patient_sex_encoded'] + \
               [col for col in df.columns if col.startswith('react_')] + \
               [col for col in df.columns if col.startswith('drug_')]

X = df[feature_cols]
y = df['serious_encoded']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Hist Gradient Boosting': HistGradientBoostingClassifier(random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    results.append({'Model': name, 'ROC AUC': roc_auc_score(y_test, y_prob)})

results_df = pd.DataFrame(results).sort_values(by='ROC AUC', ascending=False)
display(results_df)

,Model,ROC AUC
1,Random Forest,0.869718
2,Hist Gradient Boosting,0.823669
0,Logistic Regression,0.727271


## Phase 3: Advanced Optimization (Hyperparameter Tuning)
We will now take the best model (typically HistGradientBoosting or Random Forest) and find the optimal parameters using GridSearchCV.

In [26]:
print("Starting Hyperparameter Tuning for HistGradientBoosting...")

param_grid = {
    'max_iter': [100, 200],
    'max_depth': [10, 20, None],
    'learning_rate': [0.01, 0.1, 0.2],
    'min_samples_leaf': [20, 50]
}

grid_search = GridSearchCV(
    HistGradientBoostingClassifier(random_state=42),
    param_grid,
    cv=3, # 3-fold cross-validation
    scoring='roc_auc',
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train_res, y_train_res)

best_tuned_model = grid_search.best_estimator_
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV ROC AUC: {grid_search.best_score_:.4f}")

Starting Hyperparameter Tuning for HistGradientBoosting...
Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best Parameters: {'learning_rate': 0.2, 'max_depth': 20, 'max_iter': 200, 'min_samples_leaf': 20}
Best CV ROC AUC: 0.8474


## Phase 4: Final Evaluation with Cross-Validation

In [27]:
# Perform 5-Fold Cross-Validation on the TUNED model to ensure stability
cv_scores = cross_val_score(best_tuned_model, X_train_res, y_train_res, cv=5, scoring='roc_auc')
print(f"5-Fold CV ROC AUC Scores: {cv_scores}")
print(f"Mean CV ROC AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

# Final Test Evaluation
y_pred = best_tuned_model.predict(X_test_scaled)
print("\nFinal Classification Report:")
print(classification_report(y_test, y_pred))

5-Fold CV ROC AUC Scores: [0.84505137 0.84359348 0.84841087 0.85301346 0.85084414]
Mean CV ROC AUC: 0.8482 (+/- 0.0070)

Final Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.74      0.78     18479
           1       0.64      0.77      0.70     11363

    accuracy                           0.75     29842
   macro avg       0.74      0.75      0.74     29842
weighted avg       0.76      0.75      0.75     29842



## Phase 5: Serialization

In [28]:
print("Saving the FINAL TUNED model and feature names...")
joblib.dump(best_tuned_model, 'glp1_risk_model.joblib')
joblib.dump(scaler, 'scaler.joblib')
joblib.dump(le_sex, 'le_sex.joblib')
joblib.dump(feature_cols, 'feature_names.joblib') # SAVE FEATURE NAMES SEPARATELY
print("Success! Model and features are ready for deployment.")

Saving the FINAL TUNED model and feature names...
Success! Model and features are ready for deployment.
